# Inference — Autoencoder Anomaly Detection

Notebook ini melakukan prediksi anomali pada transaksi baru menggunakan model dan metadata yang sudah disimpan dari `modelling.ipynb`.

**Input yang dibutuhkan per transaksi:**
| Field | Tipe | Keterangan |
|---|---|---|
| `segmen_demografi` | str | `'Mahasiswa'` / `'First Jobber'` / `'Profesional'` |
| `kategori_detail` | str | Salah satu dari 10 kategori detail |
| `persona_encoded` | int | Tightwad=0 · Unconflicted=1 · Spendthrift=2 |
| `savings_rate` | float | Rasio tabungan bulan ini |
| `night_owl_spending` | float | Rasio transaksi malam user bulan ini |
| `survival_mode_days` | int | Hari dengan saldo < 15% gaji |
| `jam` | int | Jam transaksi (0–23), untuk hitung `is_night_owl` |
| `nominal` | float | Nilai nominal transaksi (Rupiah) |

**Output per transaksi:** `s_anomali` · `ratio` · `top_features` · `recon_error`

## 1. Import Libraries

In [35]:
import os

os.environ['TF_CPP_MIN_LOG_LEVEL']               = '2'
os.environ['TF_NUM_INTRAOP_THREADS']             = '1'
os.environ['TF_NUM_INTEROP_THREADS']             = '1'
os.environ['OBJC_DISABLE_INITIALIZE_FORK_SAFETY'] = 'YES'

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
import warnings

warnings.filterwarnings('ignore')

print(f'TensorFlow : {tf.__version__}')
print(f'Keras      : {tf.keras.__version__}')

TensorFlow : 2.21.0
Keras      : 3.14.1


## 2. Load Model & Metadata

In [36]:
MODEL_DIRECTORY = '/Users/anargyaisadhimaheswara/Documents/DBS Coding Camp/CAPSTONE/AI-ENGINEER/model'

loaded_inference_meta  = joblib.load(f'{MODEL_DIRECTORY}/anomaly_inference_meta.pkl')
best_model_name        = loaded_inference_meta['best_model_name']

dense_model = tf.keras.models.load_model(f'{MODEL_DIRECTORY}/autoencoder_anomaly.keras')
lstm_model  = tf.keras.models.load_model(f'{MODEL_DIRECTORY}/lstm_autoencoder_anomaly.keras')
best_model  = dense_model if best_model_name == 'dense' else lstm_model

print(f'Best model    : {best_model_name.upper()}')
print(f'Features      : {loaded_inference_meta["feature_columns"]}')
print()
KATEGORI_DECODING_MAP = {v: k for k, v in loaded_inference_meta['kategori_encoding_map'].items()}
print('Threshold per kategori:')
for kat_id, thr in loaded_inference_meta['threshold_per_kategori'].items():
    mean = loaded_inference_meta['mean_error_per_kategori'][kat_id]
    print(f'  [{kat_id}] {KATEGORI_DECODING_MAP[kat_id]:<30}: thr={thr:.6f}  mean={mean:.6f}')

Best model    : LSTM
Features      : ['persona_encoded', 'kategori_encoded', 'savings_rate', 'night_owl_spending', 'survival_mode_days', 'is_night_owl', 'nominal']

Threshold per kategori:
  [0] Belanja Online & Fashion      : thr=0.126813  mean=0.057337
  [1] F&B dan Nongkrong             : thr=0.126608  mean=0.046802
  [2] Groceries & Kebutuhan Pokok   : thr=0.126707  mean=0.045598
  [3] Hiburan & Langganan           : thr=0.118521  mean=0.046565
  [4] Investasi & Finansial         : thr=0.119169  mean=0.043938
  [5] Kesehatan & Perawatan Diri    : thr=0.119467  mean=0.043287
  [6] NLP Classified Transfer       : thr=0.120110  mean=0.045124
  [7] Produktivitas & Digital       : thr=0.113430  mean=0.046842
  [8] Tagihan & Utilitas            : thr=0.129847  mean=0.050021
  [9] Transportasi                  : thr=0.136148  mean=0.056749


## 3. Fungsi Prediksi

In [37]:
def predict_transaction_anomaly(transaction_dict, model, inference_meta, model_type='dense'):
    segment_name  = transaction_dict.get('segmen_demografi', 'Unknown')
    kategori_name = transaction_dict.get('kategori_detail', 'Groceries & Kebutuhan Pokok')
    kategori_id   = inference_meta['kategori_encoding_map'].get(kategori_name, 0)

    segment_scaler = inference_meta['scaler_per_segment'].get(
        segment_name, next(iter(inference_meta['scaler_per_segment'].values()))
    )
    nominal_scaler = inference_meta['scaler_nominal_per_seg_kat'].get(
        (segment_name, kategori_name),
        next(iter(inference_meta['scaler_nominal_per_seg_kat'].values()))
    )
    threshold  = inference_meta['threshold_per_kategori'].get(
        kategori_id, max(inference_meta['threshold_per_kategori'].values())
    )
    mean_error = inference_meta['mean_error_per_kategori'].get(
        kategori_id,
        sum(inference_meta['mean_error_per_kategori'].values()) / len(inference_meta['mean_error_per_kategori'])
    )

    feature_columns    = inference_meta['feature_columns']
    is_night_owl_value = int(
        (transaction_dict['jam'] >= 22) or (transaction_dict['jam'] <= 4)
    )

    raw_non_nominal = np.array([[
        transaction_dict['persona_encoded'],
        kategori_id,
        transaction_dict['savings_rate'],
        transaction_dict['night_owl_spending'],
        transaction_dict['survival_mode_days'],
        is_night_owl_value,
    ]], dtype=np.float32)
    scaled_non_nominal = segment_scaler.transform(raw_non_nominal).astype(np.float32)
    scaled_nominal     = nominal_scaler.transform(
        np.array([[transaction_dict['nominal']]], dtype=np.float32)
    ).astype(np.float32)
    scaled_features = np.hstack([scaled_non_nominal, scaled_nominal])

    if model_type == 'lstm':
        n_feat        = scaled_features.shape[1]
        input_lstm    = scaled_features.reshape(1, n_feat, 1)
        reconstructed = model.predict(input_lstm, verbose=0).reshape(1, n_feat)
    else:
        reconstructed = model.predict(scaled_features, verbose=0)

    per_feature_error = np.abs(scaled_features - reconstructed)
    total_error       = float(per_feature_error.mean())
    top_feature_names = [
        feature_columns[i] for i in np.argsort(per_feature_error[0])[::-1][:3]
    ]

    return {
        's_anomali'   : total_error > threshold,
        'ratio'       : round(total_error / (mean_error + 1e-9), 2),
        'top_features': top_feature_names,
        'recon_error' : round(total_error, 6),
    }


print(f'Fungsi siap. Model aktif: {best_model_name.upper()}')

Fungsi siap. Model aktif: LSTM


## 4. Generate 20 Data Dummy

In [38]:
dummy_transactions = [
    # ── Normal ──────────────────────────────────────────────────────────
    {'id':'D-001','segmen_demografi':'Mahasiswa',   'kategori_detail':'Groceries & Kebutuhan Pokok','persona_encoded':1,'savings_rate':0.05,'night_owl_spending':0.08,'survival_mode_days':3, 'jam':10,'nominal':75000},
    {'id':'D-002','segmen_demografi':'Mahasiswa',   'kategori_detail':'Transportasi',               'persona_encoded':1,'savings_rate':0.05,'night_owl_spending':0.08,'survival_mode_days':3, 'jam':8, 'nominal':50000},
    {'id':'D-003','segmen_demografi':'First Jobber','kategori_detail':'F&B dan Nongkrong',           'persona_encoded':1,'savings_rate':0.12,'night_owl_spending':0.10,'survival_mode_days':2, 'jam':12,'nominal':65000},
    {'id':'D-004','segmen_demografi':'First Jobber','kategori_detail':'Tagihan & Utilitas',          'persona_encoded':0,'savings_rate':0.20,'night_owl_spending':0.05,'survival_mode_days':1, 'jam':9, 'nominal':250000},
    {'id':'D-005','segmen_demografi':'Profesional', 'kategori_detail':'Investasi & Finansial',       'persona_encoded':0,'savings_rate':0.35,'night_owl_spending':0.03,'survival_mode_days':0, 'jam':11,'nominal':2000000},
    {'id':'D-006','segmen_demografi':'Profesional', 'kategori_detail':'Produktivitas & Digital',     'persona_encoded':0,'savings_rate':0.25,'night_owl_spending':0.04,'survival_mode_days':0, 'jam':10,'nominal':150000},
    {'id':'D-007','segmen_demografi':'First Jobber','kategori_detail':'Kesehatan & Perawatan Diri',  'persona_encoded':1,'savings_rate':0.10,'night_owl_spending':0.06,'survival_mode_days':3, 'jam':14,'nominal':180000},
    # ── Malam hari anomaly ────────────────────────────────────────────────
    {'id':'D-008','segmen_demografi':'Mahasiswa',   'kategori_detail':'F&B dan Nongkrong',           'persona_encoded':2,'savings_rate':-0.05,'night_owl_spending':0.40,'survival_mode_days':8, 'jam':23,'nominal':250000},
    {'id':'D-009','segmen_demografi':'First Jobber','kategori_detail':'Belanja Online & Fashion',    'persona_encoded':2,'savings_rate':-0.10,'night_owl_spending':0.35,'survival_mode_days':10,'jam':2, 'nominal':800000},
    {'id':'D-010','segmen_demografi':'Profesional', 'kategori_detail':'Hiburan & Langganan',         'persona_encoded':2,'savings_rate':-0.08,'night_owl_spending':0.30,'survival_mode_days':9, 'jam':1, 'nominal':600000},
    # ── Nominal ekstrem anomaly ───────────────────────────────────────────
    {'id':'D-011','segmen_demografi':'Mahasiswa',   'kategori_detail':'F&B dan Nongkrong',           'persona_encoded':2,'savings_rate':0.01,'night_owl_spending':0.10,'survival_mode_days':5, 'jam':14,'nominal':1500000},
    {'id':'D-012','segmen_demografi':'First Jobber','kategori_detail':'Belanja Online & Fashion',    'persona_encoded':2,'savings_rate':-0.02,'night_owl_spending':0.08,'survival_mode_days':6, 'jam':16,'nominal':3500000},
    {'id':'D-013','segmen_demografi':'Profesional', 'kategori_detail':'Tagihan & Utilitas',          'persona_encoded':1,'savings_rate':0.05,'night_owl_spending':0.06,'survival_mode_days':4, 'jam':15,'nominal':9000000},
    # ── Survival mode + savings negatif ──────────────────────────────────
    {'id':'D-014','segmen_demografi':'Mahasiswa',   'kategori_detail':'Kesehatan & Perawatan Diri',  'persona_encoded':1,'savings_rate':-0.15,'night_owl_spending':0.06,'survival_mode_days':12,'jam':13,'nominal':200000},
    {'id':'D-015','segmen_demografi':'Mahasiswa',   'kategori_detail':'Hiburan & Langganan',         'persona_encoded':2,'savings_rate':-0.20,'night_owl_spending':0.15,'survival_mode_days':14,'jam':22,'nominal':180000},
    {'id':'D-016','segmen_demografi':'First Jobber','kategori_detail':'NLP Classified Transfer',     'persona_encoded':1,'savings_rate':-0.08,'night_owl_spending':0.05,'survival_mode_days':9, 'jam':11,'nominal':150000},
    # ── Anomali kuat (multi-kondisi) ──────────────────────────────────────
    {'id':'D-017','segmen_demografi':'Mahasiswa',   'kategori_detail':'F&B dan Nongkrong',           'persona_encoded':2,'savings_rate':-0.25,'night_owl_spending':0.45,'survival_mode_days':15,'jam':1, 'nominal':2500000},
    {'id':'D-018','segmen_demografi':'First Jobber','kategori_detail':'Belanja Online & Fashion',    'persona_encoded':2,'savings_rate':-0.18,'night_owl_spending':0.38,'survival_mode_days':13,'jam':3, 'nominal':4000000},
    {'id':'D-019','segmen_demografi':'Profesional', 'kategori_detail':'Transportasi',               'persona_encoded':2,'savings_rate':-0.12,'night_owl_spending':0.30,'survival_mode_days':11,'jam':0, 'nominal':5000000},
    {'id':'D-020','segmen_demografi':'Profesional', 'kategori_detail':'Produktivitas & Digital',     'persona_encoded':0,'savings_rate':0.30,'night_owl_spending':0.02,'survival_mode_days':0, 'jam':10,'nominal':50000},
]

print(f'Total dummy transactions : {len(dummy_transactions)}')
display(pd.DataFrame(dummy_transactions)[['id','segmen_demografi','kategori_detail','nominal','jam']])

Total dummy transactions : 20


,id,segmen_demografi,kategori_detail,nominal,jam
0,D-001,Mahasiswa,Groceries & Kebutuhan Pokok,75000,10
1,D-002,Mahasiswa,Transportasi,50000,8
2,D-003,First Jobber,F&B dan Nongkrong,65000,12
3,D-004,First Jobber,Tagihan & Utilitas,250000,9
4,D-005,Profesional,Investasi & Finansial,2000000,11
5,D-006,Profesional,Produktivitas & Digital,150000,10
6,D-007,First Jobber,Kesehatan & Perawatan Diri,180000,14
7,D-008,Mahasiswa,F&B dan Nongkrong,250000,23
8,D-009,First Jobber,Belanja Online & Fashion,800000,2
9,D-010,Profesional,Hiburan & Langganan,600000,1


## 5. Prediksi dengan Best Model & Bangun RAG JSON

In [ ]:
MEAN_NOMINAL_BASELINE = {
    'Belanja Online & Fashion'    : 189561,
    'F&B dan Nongkrong'           : 69202,
    'Groceries & Kebutuhan Pokok' : 80657,
    'Hiburan & Langganan'         : 134237,
    'Investasi & Finansial'       : 635739,
    'Kesehatan & Perawatan Diri'  : 174004,
    'NLP Classified Transfer'     : 78307,
    'Produktivitas & Digital'     : 42260,
    'Tagihan & Utilitas'          : 465869,
    'Transportasi'                : 169952,
}

user_baselines = {'mean_nominal': MEAN_NOMINAL_BASELINE}


def build_anomaly_context(row, user_baselines):
    """Jabarkan SEMUA top_features ke dalam kalimat konteks."""
    ctx = []
    kat = row['kategori_detail']

    for feat in row['top_features']:

        if feat == 'nominal':
            mean_nom  = user_baselines['mean_nominal'].get(kat, 1)
            ratio_nom = row['nominal'] / mean_nom
            ctx.append(
                f"Nominal transaksi Rp {row['nominal']:,.0f} — "
                f"{ratio_nom:.1f}x dari rata-rata {kat} "
                f"(Rp {mean_nom:,.0f})"
            )

        elif feat == 'savings_rate':
            sr = row['savings_rate']
            if sr < 0:
                ctx.append(
                    f"Savings rate bulan ini {sr:.1%} (negatif) — "
                    f"pengeluaran melebihi pemasukan"
                )
            else:
                ctx.append(
                    f"Savings rate bulan ini {sr:.1%} — "
                    f"lebih rendah dari pola normalmu"
                )

        elif feat == 'night_owl_spending':
            now = row['night_owl_spending']
            ctx.append(
                f"Rasio pengeluaran malam hari bulan ini {now:.1%} — "
                f"lebih tinggi dari pola normalmu"
            )

        elif feat == 'is_night_owl':
            ctx.append(
                "Transaksi ini terjadi dini hari — pola yang tidak biasa untukmu"
            )

        elif feat == 'survival_mode_days':
            days = int(row['survival_mode_days'])
            ctx.append(
                f"Sudah {days} hari saldo di bawah 15% gaji bulan ini"
            )

        elif feat == 'kategori_encoded':
            mean_nom  = user_baselines['mean_nominal'].get(kat, 1)
            ratio_nom = row['nominal'] / mean_nom
            ctx.append(
                f"Pola pengeluaran pada kategori '{kat}' tidak biasa — "
                f"nominal Rp {row['nominal']:,.0f} adalah {ratio_nom:.1f}x "
                f"dari rata-rata kategori ini (Rp {mean_nom:,.0f})"
            )

        elif feat == 'persona_encoded':
            ctx.append(
                "Perilaku belanja berubah dari persona biasamu"
            )

    return " | ".join(ctx) if ctx else ""


# Prediksi semua 20 transaksi dummy
rag_records = []
for trx in dummy_transactions:
    pred = predict_transaction_anomaly(
        trx, best_model, loaded_inference_meta, model_type=best_model_name
    )

    row_for_ctx = {
        'nominal'            : trx['nominal'],
        'savings_rate'       : trx['savings_rate'],
        'night_owl_spending' : trx['night_owl_spending'],
        'survival_mode_days' : trx['survival_mode_days'],
        'persona_encoded'    : trx['persona_encoded'],
        'kategori_detail'    : trx['kategori_detail'],
        'top_features'       : pred['top_features'],
    }

    rag_records.append({
        'id_transaksi'     : trx['id'],
        'segmen_demografi' : trx['segmen_demografi'],
        'kategori_detail'  : trx['kategori_detail'],
        'nominal'          : trx['nominal'],
        'jam'              : trx['jam'],
        's_anomali'        : bool(pred['s_anomali']),
        'ratio'            : pred['ratio'],
        'recon_error'      : pred['recon_error'],
        'top_features'     : pred['top_features'],
        'anomaly_context'  : build_anomaly_context(row_for_ctx, user_baselines) if pred['s_anomali'] else '',
    })

df_results = pd.DataFrame(rag_records)
print(f'Total transaksi  : {len(df_results)}')
print(f'Anomali          : {df_results["s_anomali"].sum()}')
print(f'Normal           : {(~df_results["s_anomali"]).sum()}')
print()
display(df_results[['id_transaksi','kategori_detail','nominal','s_anomali','ratio','top_features','anomaly_context']])

Total transaksi  : 20
Anomali          : 16
Normal           : 4



,id_transaksi,kategori_detail,nominal,s_anomali,ratio,top_features,anomaly_context
0,D-001,Groceries & Kebutuhan Pokok,75000,False,0.67,"[persona_encoded, savings_rate, kategori_encoded]",
1,D-002,Transportasi,50000,False,0.93,"[savings_rate, kategori_encoded, persona_encoded]",
2,D-003,F&B dan Nongkrong,65000,True,2.91,"[night_owl_spending, savings_rate, kategori_en...",Rasio pengeluaran malam hari bulan ini 10.0% —...
3,D-004,Tagihan & Utilitas,250000,True,4.61,"[savings_rate, survival_mode_days, kategori_en...",Savings rate bulan ini 20.0% — pola tabungan b...
4,D-005,Investasi & Finansial,2000000,True,26.60,"[savings_rate, nominal, survival_mode_days]",Savings rate bulan ini 35.0% — pola tabungan b...
5,D-006,Produktivitas & Digital,150000,True,19.61,"[savings_rate, nominal, survival_mode_days]",Savings rate bulan ini 25.0% — pola tabungan b...
6,D-007,Kesehatan & Perawatan Diri,180000,False,0.97,"[savings_rate, survival_mode_days, kategori_en...",
7,D-008,F&B dan Nongkrong,250000,True,22.13,"[night_owl_spending, survival_mode_days, nominal]",Rasio pengeluaran malam hari bulan ini 40.0% —...
8,D-009,Belanja Online & Fashion,800000,True,21.47,"[night_owl_spending, survival_mode_days, savin...",Rasio pengeluaran malam hari bulan ini 35.0% —...
9,D-010,Hiburan & Langganan,600000,True,38.46,"[night_owl_spending, nominal, savings_rate]",Rasio pengeluaran malam hari bulan ini 30.0% —...


## 6. Export RAG JSON

In [40]:
import json as json_lib

DATA_DIRECTORY = '/Users/anargyaisadhimaheswara/Documents/DBS Coding Camp/CAPSTONE/AI-ENGINEER/data'
OUTPUT_PATH    = f'{DATA_DIRECTORY}/anomaly_inference_results.json'

with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json_lib.dump(rag_records, f, indent=2, ensure_ascii=False)

rag_json = json_lib.dumps(rag_records, indent=2, ensure_ascii=False)
print(rag_json[:3000], '...') if len(rag_json) > 3000 else print(rag_json)
print(f'[Total: {len(rag_records)} | Anomali: {sum(r["s_anomali"] for r in rag_records)} | Normal: {sum(not r["s_anomali"] for r in rag_records)}]')
print(f'Saved → {OUTPUT_PATH}')

[
  {
    "id_transaksi": "D-001",
    "segmen_demografi": "Mahasiswa",
    "kategori_detail": "Groceries & Kebutuhan Pokok",
    "nominal": 75000,
    "jam": 10,
    "s_anomali": false,
    "ratio": 0.67,
    "recon_error": 0.030675,
    "top_features": [
      "persona_encoded",
      "savings_rate",
      "kategori_encoded"
    ],
    "anomaly_context": ""
  },
  {
    "id_transaksi": "D-002",
    "segmen_demografi": "Mahasiswa",
    "kategori_detail": "Transportasi",
    "nominal": 50000,
    "jam": 8,
    "s_anomali": false,
    "ratio": 0.93,
    "recon_error": 0.052978,
    "top_features": [
      "savings_rate",
      "kategori_encoded",
      "persona_encoded"
    ],
    "anomaly_context": ""
  },
  {
    "id_transaksi": "D-003",
    "segmen_demografi": "First Jobber",
    "kategori_detail": "F&B dan Nongkrong",
    "nominal": 65000,
    "jam": 12,
    "s_anomali": true,
    "ratio": 2.91,
    "recon_error": 0.136338,
    "top_features": [
      "night_owl_spending",
      "sa